# Exploración de datos de Instacart con Python y Pandas


**Actividad:** GA1-220501094-01-AA1-EV04 – Fundamentos en Pandas · *Explorando Instacart: descubre patrones de compra*  
**Rol:** analista de datos de Instacart (desarrollo individual, Aprendizaje Basado en Problemas)  
**Aprendiz:** Juan Camilo Sarrazola Agudelo

> **Cómo está construido este notebook:** las cifras de las interpretaciones no están escritas a mano.
> Cada interpretación se calcula con código a partir de los DataFrames, de modo que las respuestas
> siempre corresponden a los datos que se cargaron.

---
## Momento 1. Comprensión del problema y reconocimiento de los datos

### 1.1 Descripción del problema
Instacart es una plataforma de compras de supermercado. Cada vez que un cliente realiza un pedido queda
registrado quién compró, cuándo lo hizo, qué productos incluyó y si ya los había comprado antes.
El equipo de operaciones necesita saber **qué información contienen los archivos** y **cómo consultarla**
para responder preguntas sobre el comportamiento de los clientes.

**Pregunta orientadora:** ¿cómo utilizar Python y Pandas para explorar los datos de Instacart, identificar las
características de los pedidos y productos, y obtener información relevante sobre los hábitos de compra de sus clientes?

### 1.2 Los cinco archivos y la información que almacenan

| N.º | Archivo | DataFrame | Qué información se espera que almacene |
|---|---|---|---|
| 1 | `instacart_orders.csv` | `orders` | Un registro por pedido: quién lo hizo (`user_id`), el número de pedido de ese usuario, día de la semana, hora y días transcurridos desde el pedido anterior. |
| 2 | `products.csv` | `products` | Catálogo de productos: identificador, nombre, pasillo y departamento al que pertenece. |
| 3 | `aisles.csv` | `aisles` | Tabla de consulta: nombre de cada pasillo (`aisle_id` → `aisle`). |
| 4 | `departments.csv` | `departments` | Tabla de consulta: nombre de cada departamento (`department_id` → `department`). |
| 5 | `order_products.csv` | `order_products` | Detalle de cada pedido: un registro por producto incluido, su posición en el carrito y si es una recompra. |

### 1.3 Columnas que identifican pedidos, usuarios y productos

| Qué identifica | Columna | Dónde aparece |
|---|---|---|
| Pedidos | `order_id` | `orders` y `order_products` |
| Usuarios (clientes) | `user_id` | solo en `orders` |
| Productos | `product_id` | `products` y `order_products` |
| Pasillos | `aisle_id` | `products` y `aisles` |
| Departamentos | `department_id` | `products` y `departments` |

### 1.4 Tres preguntas de análisis sobre hábitos de compra
Se formulan de modo que puedan resolverse con las operaciones básicas de la actividad (filtrado, conteo, suma y promedio):

1. **¿Cada cuántos días, en promedio, vuelven a comprar los clientes?** (promedio de `days_since_prior_order`).
2. **¿Qué proporción de los artículos de los pedidos son recompras?** (filtrado y conteo de `reordered`).
3. **¿Cuántos pedidos se realizan a las 10 horas y corresponden a clientes con más de 10 pedidos?** (filtro con dos condiciones simultáneas sobre `order_hour_of_day` y `order_number`).

### 1.5 ¿Por qué explorar los datos antes de analizarlos?
Explorar primero permite saber **cuántos registros hay, qué significa cada columna, de qué tipo es y si hay valores ausentes**.
Sin ese paso es fácil cometer errores: confundir filas con clientes, promediar columnas con valores faltantes sin darse cuenta,
filtrar por una columna que contiene texto en lugar de números o unir tablas por una clave equivocada.
La exploración inicial es la base que hace confiables todos los análisis posteriores.

---
## Momento 2. Importación y exploración inicial de los archivos

In [1]:
import pandas as pd
from pathlib import Path
from IPython.display import display, Markdown

pd.set_option("display.max_columns", None)


def md(texto):
    """Muestra un texto en formato Markdown dentro del notebook."""
    display(Markdown(texto))


def fmt(n):
    """Entero con punto de miles (estilo es-CO): 478967 -> 478.967"""
    return f"{n:,.0f}".replace(",", ".")


def fmt2(x):
    """Decimal con coma decimal y punto de miles: 17.1 -> 17,10"""
    return f"{x:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")

### 2.1 Carga de los cinco archivos
Los archivos usan **punto y coma (`;`)** como separador, por eso se indica `sep=";"`.
La ruta asume la estructura del repositorio (`notebooks/` y `data/` son carpetas hermanas).
Si trabaja en Google Colab, cambie `RUTA_DATOS` por la ruta de su Drive, por ejemplo `Path("/content/drive/MyDrive/instacart")`.

In [2]:
# Ajuste esta ruta según su entorno de trabajo
RUTA_DATOS = Path("../data")
if not RUTA_DATOS.exists():
    RUTA_DATOS = Path("data")
if not RUTA_DATOS.exists():
    RUTA_DATOS = Path(".")

orders = pd.read_csv(RUTA_DATOS / "instacart_orders.csv", sep=";")
products = pd.read_csv(RUTA_DATOS / "products.csv", sep=";")
aisles = pd.read_csv(RUTA_DATOS / "aisles.csv", sep=";")
departments = pd.read_csv(RUTA_DATOS / "departments.csv", sep=";")
order_products = pd.read_csv(RUTA_DATOS / "order_products.csv", sep=";")

# Diccionario para recorrer los cinco DataFrames con un solo bucle
dataframes = {
    "orders": orders,
    "products": products,
    "aisles": aisles,
    "departments": departments,
    "order_products": order_products,
}
print("Archivos cargados desde:", RUTA_DATOS.resolve())

Archivos cargados desde: C:\Users\El Sarra\Desktop\instacart-analisis\data


### 2.2 Primeros y últimos cinco registros de cada DataFrame (`head()` y `tail()`)

In [3]:
for nombre, df in dataframes.items():
    md(f"#### `{nombre}` · primeros 5 registros")
    display(df.head())
    md(f"#### `{nombre}` · últimos 5 registros")
    display(df.tail())

#### `orders` · primeros 5 registros

,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,1515936,183418,11,6,13,30.0
1,1690866,163593,5,5,12,9.0
2,1454967,39980,4,5,19,2.0
3,1768857,82516,56,0,20,10.0
4,3007858,196724,2,4,12,17.0


#### `orders` · últimos 5 registros

,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
478962,3210681,5617,5,1,14,7.0
478963,3270802,112087,2,3,13,6.0
478964,885349,82944,16,2,11,6.0
478965,216274,4391,3,3,8,8.0
478966,2071924,1730,18,1,14,15.0


#### `products` · primeros 5 registros

...

,department_id,department
16,17,household
17,18,babies
18,19,snacks
19,20,deli
20,21,missing


#### `order_products` · primeros 5 registros

,order_id,product_id,add_to_cart_order,reordered
0,2141543,11440,17.0,0
1,567889,1560,1.0,1
2,2261212,26683,1.0,1
3,491251,8670,35.0,1
4,2571142,1940,5.0,1


#### `order_products` · últimos 5 registros

,order_id,product_id,add_to_cart_order,reordered
4545002,577211,15290,12.0,1
4545003,1219554,21914,9.0,0
4545004,692640,47766,4.0,1
4545005,319435,691,8.0,1
4545006,1398151,28733,9.0,0


### 2.3 Dimensiones de cada DataFrame (`shape`)

In [4]:
for nombre, df in dataframes.items():
    print(f"{nombre:15s} shape = {df.shape}")

resumen_dimensiones = pd.DataFrame(
    {
        "DataFrame": list(dataframes.keys()),
        "filas": [df.shape[0] for df in dataframes.values()],
        "columnas": [df.shape[1] for df in dataframes.values()],
    }
)
resumen_dimensiones

orders          shape = (478967, 6)
products        shape = (49694, 4)
aisles          shape = (134, 2)
departments     shape = (21, 2)
order_products  shape = (4545007, 4)


,DataFrame,filas,columnas
0,orders,478967,6
1,products,49694,4
2,aisles,134,2
3,departments,21,2
4,order_products,4545007,4


### 2.4 Preguntas de análisis del Momento 2

In [5]:
mas_filas = resumen_dimensiones.loc[resumen_dimensiones["filas"].idxmax()]
min_cols = resumen_dimensiones["columnas"].min()
menos_columnas = resumen_dimensiones.loc[resumen_dimensiones["columnas"] == min_cols, "DataFrame"].tolist()
menos_columnas_txt = " y ".join(f"`{n}`" for n in menos_columnas)

md(f'''
**1. ¿Cuál de los cinco DataFrames contiene más registros?**
`{mas_filas["DataFrame"]}`, con **{fmt(mas_filas["filas"])}** filas.

**2. ¿Cuál contiene menos columnas?**
{menos_columnas_txt}, con **{min_cols}** columnas cada uno (son tablas de consulta: un identificador y un nombre).

**3. ¿Qué diferencias identifica entre los datos de pedidos y los datos de productos?**
`orders` tiene **{fmt(orders.shape[0])}** registros y {orders.shape[1]} columnas; describe *eventos*: cuándo y con qué frecuencia compra cada usuario
(día, hora, número de pedido, días desde el pedido anterior). `products` tiene **{fmt(products.shape[0])}** registros y {products.shape[1]} columnas;
describe *objetos*: el catálogo, con el nombre del producto y su ubicación en pasillos y departamentos.
Los pedidos cambian con cada compra; el catálogo solo cambia cuando se agregan o retiran productos.

**4. ¿Por qué es importante conocer las dimensiones de un conjunto de datos?**
Porque define cuánta memoria y tiempo exigirá el análisis (aquí `order_products` es el más pesado),
permite comprobar que el archivo se cargó completo y que el separador fue el correcto (si `sep` estuviera mal, saldría **una sola columna**),
y ayuda a interpretar cualquier conteo posterior frente al total de filas.
''')


**1. ¿Cuál de los cinco DataFrames contiene más registros?**
`order_products`, con **4.545.007** filas.

**2. ¿Cuál contiene menos columnas?**
`aisles` y `departments`, con **2** columnas cada uno (son tablas de consulta: un identificador y un nombre).

**3. ¿Qué diferencias identifica entre los datos de pedidos y los datos de productos?**
`orders` tiene **478.967** registros y 6 columnas; describe *eventos*: cuándo y con qué frecuencia compra cada usuario
(día, hora, número de pedido, días desde el pedido anterior). `products` tiene **49.694** registros y 4 columnas;
describe *objetos*: el catálogo, con el nombre del producto y su ubicación en pasillos y departamentos.
Los pedidos cambian con cada compra; el catálogo solo cambia cuando se agregan o retiran productos.

**4. ¿Por qué es importante conocer las dimensiones de un conjunto de datos?**
Porque define cuánta memoria y tiempo exigirá el análisis (aquí `order_products` es el más pesado),
permite comprobar que el archivo se cargó completo y que el separador fue el correcto (si `sep` estuviera mal, saldría **una sola columna**),
y ayuda a interpretar cualquier conteo posterior frente al total de filas.


---
## Momento 3. Reconocimiento de la estructura y calidad inicial de los datos
En esta fase **no se corrige ni se elimina ningún registro**: solo se reconocen las características y limitaciones de los datos.

### 3.1 Columnas, tipos de datos e `info()` de cada DataFrame

In [6]:
for nombre, df in dataframes.items():
    print("=" * 70)
    print(f"DataFrame: {nombre}   |   shape = {df.shape}")
    print("Columnas:", list(df.columns))
    print()
    print("dtypes:")
    print(df.dtypes)
    print()
    df.info()
    print()

DataFrame: orders   |   shape = (478967, 6)
Columnas: ['order_id', 'user_id', 'order_number', 'order_dow', 'order_hour_of_day', 'days_since_prior_order']

dtypes:
order_id                    int64
user_id                     int64
order_number                int64
order_dow                   int64
order_hour_of_day           int64
days_since_prior_order    float64
dtype: object

<class 'pandas.DataFrame'>
RangeIndex: 478967 entries, 0 to 478966
Data columns (total 6 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   order_id                478967 non-null  int64  
 1   user_id                 478967 non-null  int64  
 2   order_number            478967 non-null  int64  
 3   order_dow               478967 non-null  int64  
 4   order_hour_of_day       478967 non-null  int64  
 5   days_since_prior_order  450148 non-null  float64
dtypes: float64(1), int64(5)
memory usage: 21.9 MB

DataFrame: products   |   shape = (

### 3.2 Tipo de cada columna: entero, decimal o texto

In [7]:
def clasificar(dtype):
    if pd.api.types.is_integer_dtype(dtype):
        return "entero"
    if pd.api.types.is_float_dtype(dtype):
        return "decimal"
    return "texto"


tipos = pd.DataFrame(
    [
        {"DataFrame": nombre, "columna": col, "dtype": str(df[col].dtype), "tipo": clasificar(df[col].dtype)}
        for nombre, df in dataframes.items()
        for col in df.columns
    ]
)
tipos

,DataFrame,columna,dtype,tipo
0,orders,order_id,int64,entero
1,orders,user_id,int64,entero
2,orders,order_number,int64,entero
3,orders,order_dow,int64,entero
4,orders,order_hour_of_day,int64,entero
5,orders,days_since_prior_order,float64,decimal
6,products,product_id,int64,entero
7,products,product_name,str,texto
8,products,aisle_id,int64,entero
9,products,department_id,int64,entero


### 3.3 Campos que relacionan conceptualmente los conjuntos de datos
Un mismo identificador aparece en dos tablas y permite unirlas. El código verifica que cada columna clave exista realmente en ambos DataFrames.

In [8]:
relaciones = [
    ("orders", "order_products", "order_id", "Une cada pedido con los productos que contiene"),
    ("products", "order_products", "product_id", "Une cada producto del catálogo con sus apariciones en pedidos"),
    ("products", "aisles", "aisle_id", "Asigna el nombre del pasillo a cada producto"),
    ("products", "departments", "department_id", "Asigna el nombre del departamento a cada producto"),
]

filas_rel = []
for izquierda, derecha, clave, uso in relaciones:
    en_ambas = clave in dataframes[izquierda].columns and clave in dataframes[derecha].columns
    filas_rel.append({"tabla A": izquierda, "tabla B": derecha, "campo en común": clave,
                      "existe en ambas": en_ambas, "para qué sirve": uso})
pd.DataFrame(filas_rel)

,tabla A,tabla B,campo en común,existe en ambas,para qué sirve
0,orders,order_products,order_id,True,Une cada pedido con los productos que contiene
1,products,order_products,product_id,True,Une cada producto del catálogo con sus aparici...
2,products,aisles,aisle_id,True,Asigna el nombre del pasillo a cada producto
3,products,departments,department_id,True,Asigna el nombre del departamento a cada producto


### 3.4 y 3.5 Valores no nulos frente al número de registros
`info()` informa cuántos valores **no nulos** tiene cada columna. Si ese número es menor que el total de filas, la diferencia son valores ausentes.

In [9]:
filas_calidad = []
for nombre, df in dataframes.items():
    for col in df.columns:
        no_nulos = int(df[col].notna().sum())
        ausentes = len(df) - no_nulos
        filas_calidad.append({
            "DataFrame": nombre,
            "columna": col,
            "registros": len(df),
            "no_nulos": no_nulos,
            "ausentes": ausentes,
            "pct_ausentes": round(100 * ausentes / len(df), 3),
        })
calidad = pd.DataFrame(filas_calidad)

md("**Columnas con valores ausentes:**")
display(calidad[calidad["ausentes"] > 0])

md("**Detalle completo, columna por columna:**")
display(calidad)

**Columnas con valores ausentes:**

,DataFrame,columna,registros,no_nulos,ausentes,pct_ausentes
5,orders,days_since_prior_order,478967,450148,28819,6.017
7,products,product_name,49694,48436,1258,2.531
16,order_products,add_to_cart_order,4545007,4544171,836,0.018


**Detalle completo, columna por columna:**

,DataFrame,columna,registros,no_nulos,ausentes,pct_ausentes
0,orders,order_id,478967,478967,0,0.000
1,orders,user_id,478967,478967,0,0.000
2,orders,order_number,478967,478967,0,0.000
3,orders,order_dow,478967,478967,0,0.000
4,orders,order_hour_of_day,478967,478967,0,0.000
5,orders,days_since_prior_order,478967,450148,28819,6.017
6,products,product_id,49694,49694,0,0.000
7,products,product_name,49694,48436,1258,2.531
8,products,aisle_id,49694,49694,0,0.000
9,products,department_id,49694,49694,0,0.000


### 3.6 Tabla de diagnóstico

In [10]:
filas_diag = []
for nombre, df in dataframes.items():
    cols_con_nulos = calidad[(calidad["DataFrame"] == nombre) & (calidad["ausentes"] > 0)]
    if cols_con_nulos.empty:
        obs = "Sin valores ausentes en ninguna columna."
    else:
        detalle = "; ".join(f"{r.columna}: {fmt(r.ausentes)} ausentes ({r.pct_ausentes:.2f} %)"
                            for r in cols_con_nulos.itertuples(index=False))
        obs = f"Valores ausentes → {detalle}."
    filas_diag.append({
        "Archivo": f"{nombre}.csv" if nombre != "orders" else "instacart_orders.csv",
        "Filas": fmt(df.shape[0]),
        "Columnas": df.shape[1],
        "Observaciones": obs,
    })

diagnostico = pd.DataFrame(filas_diag)
pd.set_option("display.max_colwidth", None)
diagnostico

,Archivo,Filas,Columnas,Observaciones
0,instacart_orders.csv,478.967,6,Valores ausentes → days_since_prior_order: 28.819 ausentes (6.02 %).
1,products.csv,49.694,4,Valores ausentes → product_name: 1.258 ausentes (2.53 %).
2,aisles.csv,134,2,Sin valores ausentes en ninguna columna.
3,departments.csv,21,2,Sin valores ausentes en ninguna columna.
4,order_products.csv,4.545.007,4,Valores ausentes → add_to_cart_order: 836 ausentes (0.02 %).


### 3.7 Preguntas de análisis del Momento 3

In [11]:
cols_nulas = calidad[calidad["ausentes"] > 0]
if cols_nulas.empty:
    texto_nulas = "No se observan valores ausentes en ninguna columna."
else:
    texto_nulas = "; ".join(
        f"`{r.DataFrame}.{r.columna}` ({fmt(r.ausentes)} ausentes de {fmt(r.registros)})"
        for r in cols_nulas.itertuples(index=False)
    )

# Los ausentes de days_since_prior_order ¿corresponden al primer pedido del usuario?
ausentes_dias = orders[orders["days_since_prior_order"].isna()]
if len(ausentes_dias) > 0 and (ausentes_dias["order_number"] == 1).all():
    explicacion_dias = ("Todos los ausentes de `days_since_prior_order` tienen `order_number` = 1: es el primer pedido del usuario "
                        "y no existe un pedido anterior, de modo que la ausencia tiene una explicación lógica.")
else:
    explicacion_dias = "Conviene revisar si los ausentes de `days_since_prior_order` corresponden a primeros pedidos."

md(f'''
**1. ¿Qué diferencias existen entre `int64`, `float64` y `object`?**
- `int64`: números enteros, sin decimales (identificadores, horas, contadores). **No admite valores ausentes**.
- `float64`: números con decimales. También es el tipo que Pandas usa cuando una columna entera tiene valores ausentes, porque `NaN` es un decimal.
  Por eso `days_since_prior_order` y `add_to_cart_order` aparecen como `float64`: se ven como `30.0` o `1.0`.
- `object`: texto (cadenas), como `product_name`, `aisle` y `department`.

**2. ¿Qué información proporciona `info()` que no se obtiene directamente con `shape`?**
`shape` solo da filas y columnas. `info()` agrega el **nombre y tipo de cada columna**, el **número de valores no nulos** (para detectar ausentes) y el **uso de memoria**.

**3. ¿En qué columnas se observan valores ausentes?**
{texto_nulas}

{explicacion_dias}

**4. ¿Qué dificultades podrían generar estos valores en un análisis posterior?**
- Un promedio o conteo puede basarse en menos registros de los que parece (`mean()` ignora los ausentes).
- Un filtro como `days_since_prior_order > 15` **excluye en silencio** las filas ausentes, porque `NaN` no es mayor ni menor que nada.
- Las columnas con ausentes pasan a `float64`, lo que puede sorprender al leer resultados (`1.0` en vez de `1`).
- Un texto ausente (por ejemplo, un nombre de producto) puede producir errores al aplicar métodos de texto.
''')


**1. ¿Qué diferencias existen entre `int64`, `float64` y `object`?**
- `int64`: números enteros, sin decimales (identificadores, horas, contadores). **No admite valores ausentes**.
- `float64`: números con decimales. También es el tipo que Pandas usa cuando una columna entera tiene valores ausentes, porque `NaN` es un decimal.
  Por eso `days_since_prior_order` y `add_to_cart_order` aparecen como `float64`: se ven como `30.0` o `1.0`.
- `object`: texto (cadenas), como `product_name`, `aisle` y `department`.

**2. ¿Qué información proporciona `info()` que no se obtiene directamente con `shape`?**
`shape` solo da filas y columnas. `info()` agrega el **nombre y tipo de cada columna**, el **número de valores no nulos** (para detectar ausentes) y el **uso de memoria**.

**3. ¿En qué columnas se observan valores ausentes?**
`orders.days_since_prior_order` (28.819 ausentes de 478.967); `products.product_name` (1.258 ausentes de 49.694); `order_products.add_to_cart_order` (836 ausentes de 4.545.007)

Todos los ausentes de `days_since_prior_order` tienen `order_number` = 1: es el primer pedido del usuario y no existe un pedido anterior, de modo que la ausencia tiene una explicación lógica.

**4. ¿Qué dificultades podrían generar estos valores en un análisis posterior?**
- Un promedio o conteo puede basarse en menos registros de los que parece (`mean()` ignora los ausentes).
- Un filtro como `days_since_prior_order > 15` **excluye en silencio** las filas ausentes, porque `NaN` no es mayor ni menor que nada.
- Las columnas con ausentes pasan a `float64`, lo que puede sorprender al leer resultados (`1.0` en vez de `1`).
- Un texto ausente (por ejemplo, un nombre de producto) puede producir errores al aplicar métodos de texto.


---
## Momento 4. Indexación y filtrado lógico de la información
Cada ejercicio incluye el **código**, la **salida** (número de registros y primeras filas) y una **explicación breve**.
Se usa `loc[]` para que la selección de filas y columnas quede explícita.

In [12]:
def mostrar(resultado, titulo):
    """Imprime cuántos registros devolvió la consulta y sus primeras filas."""
    md(f"**{titulo}** → **{fmt(len(resultado))}** registros · {resultado.shape[1]} columnas")
    display(resultado.head())

### Ejercicio 1 · `products`: seleccionar solo `product_id` y `product_name`

In [13]:
ej1 = products.loc[:, ["product_id", "product_name"]]
mostrar(ej1, "Ejercicio 1")

**Ejercicio 1** → **49.694** registros · 2 columnas

,product_id,product_name
0,1,Chocolate Sandwich Cookies
1,2,All-Seasons Salt
2,3,Robust Golden Unsweetened Oolong Tea
3,4,Smart Ones Classic Favorites Mini Rigatoni With Vodka Cream Sauce
4,5,Green Chile Anytime Sauce


**Explicación:** `:` toma *todas* las filas y la lista de nombres elige solo dos columnas. Cambia el ancho de la tabla, no el número de registros.

### Ejercicio 2 · `products`: productos con `department_id` igual a 1

In [14]:
ej2 = products.loc[products["department_id"] == 1]
mostrar(ej2, "Ejercicio 2")

**Ejercicio 2** → **4.008** registros · 4 columnas

,product_id,product_name,aisle_id,department_id
3,4,Smart Ones Classic Favorites Mini Rigatoni With Vodka Cream Sauce,38,1
7,8,Cut Russet Potatoes Steam N' Mash,116,1
11,12,Chocolate Fudge Layer Cake,119,1
17,18,Pizza for One Suprema Frozen Pizza,79,1
29,30,Three Cheese Ziti- Marinara with Meatballs,38,1


**Explicación:** la comparación `== 1` crea una serie de verdaderos/falsos y `loc` conserva solo las filas verdaderas. Se mantienen todas las columnas.

### Ejercicio 3 · `orders`: pedidos realizados a las 10 horas

In [15]:
ej3 = orders.loc[orders["order_hour_of_day"] == 10]
mostrar(ej3, "Ejercicio 3")

**Ejercicio 3** → **40.578** registros · 6 columnas

,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
19,294269,107523,29,2,10,1.0
28,133707,182261,1,3,10,NaN
36,3378084,127629,21,1,10,20.0
49,2161799,87345,7,0,10,24.0
51,1739741,118405,48,2,10,3.0


**Explicación:** filtra por la hora del día (0 a 23). Cada fila que aparece es un pedido realizado a las 10 h.

### Ejercicio 4 · `orders`: pedidos con `order_dow` igual a 0

In [16]:
ej4 = orders.loc[orders["order_dow"] == 0]
mostrar(ej4, "Ejercicio 4")

**Ejercicio 4** → **84.090** registros · 6 columnas

,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
3,1768857,82516,56,0,20,10.0
5,1843871,20084,5,0,11,15.0
7,2521283,97711,2,0,18,22.0
23,3406282,19356,21,0,0,14.0
25,1895419,169793,2,0,11,8.0


**Explicación:** filtra por el código 0 de `order_dow` (día de la semana). El archivo no trae un diccionario que diga a qué día corresponde cada código, así que no se asume ninguno.

### Ejercicio 5 · `orders`: pedidos con `order_number` superior a 10

In [17]:
ej5 = orders.loc[orders["order_number"] > 10]
mostrar(ej5, "Ejercicio 5")

**Ejercicio 5** → **244.072** registros · 6 columnas

,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,1515936,183418,11,6,13,30.0
3,1768857,82516,56,0,20,10.0
6,2511640,148233,40,3,8,2.0
8,2266494,34954,40,5,15,4.0
10,1162566,43738,33,4,11,2.0


**Explicación:** `order_number` cuenta cuántos pedidos lleva el usuario. Un valor mayor a 10 identifica pedidos de clientes con historial de compra más largo.

### Ejercicio 6 · `orders`: pedidos con `days_since_prior_order` superior a 15

In [18]:
ej6 = orders.loc[orders["days_since_prior_order"] > 15]
mostrar(ej6, "Ejercicio 6")

**Ejercicio 6** → **110.706** registros · 6 columnas

,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,1515936,183418,11,6,13,30.0
4,3007858,196724,2,4,12,17.0
7,2521283,97711,2,0,18,22.0
14,2843219,74699,4,6,11,30.0
26,1844247,122558,4,6,18,30.0


**Explicación:** muestra pedidos de clientes que tardaron más de 15 días en volver a comprar. Las filas con valor ausente **no** entran en el resultado, porque `NaN > 15` es falso.

### Ejercicio 7 · `order_products`: registros con `reordered` igual a 1

In [19]:
ej7 = order_products.loc[order_products["reordered"] == 1]
mostrar(ej7, "Ejercicio 7")

**Ejercicio 7** → **2.683.838** registros · 4 columnas

,order_id,product_id,add_to_cart_order,reordered
1,567889,1560,1.0,1
2,2261212,26683,1.0,1
3,491251,8670,35.0,1
4,2571142,1940,5.0,1
5,2456893,21616,4.0,1


**Explicación:** `reordered` vale 1 cuando el cliente ya había comprado ese producto antes (recompra) y 0 cuando es la primera vez. El resultado son solo las recompras.

### Ejercicio 8 · `order_products`: productos en la primera posición del carrito

In [20]:
ej8 = order_products.loc[order_products["add_to_cart_order"] == 1]
mostrar(ej8, "Ejercicio 8")

**Ejercicio 8** → **450.046** registros · 4 columnas

,order_id,product_id,add_to_cart_order,reordered
1,567889,1560,1.0,1
2,2261212,26683,1.0,1
14,1961225,37553,1.0,1
16,639939,10017,1.0,1
23,750040,8518,1.0,0


**Explicación:** `add_to_cart_order` es el orden en que se agregó cada producto. El valor 1 corresponde al primer producto de cada pedido. Aunque se vea como `1.0`, la comparación con `1` funciona.

### Filtros con condiciones simultáneas
Cuando hay dos condiciones se usa **`&`** (y) con **paréntesis** en cada una. En Pandas no se usan las palabras `and` / `or`.

### Ejercicio 9 · `orders`: pedidos a las 10 horas **y** con `order_number` superior a 10

In [21]:
ej9 = orders.loc[(orders["order_hour_of_day"] == 10) & (orders["order_number"] > 10)]
mostrar(ej9, "Ejercicio 9")

**Ejercicio 9** → **21.537** registros · 6 columnas

,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
19,294269,107523,29,2,10,1.0
36,3378084,127629,21,1,10,20.0
51,1739741,118405,48,2,10,3.0
92,60724,110808,78,0,10,2.0
140,3156095,113920,14,2,10,2.0


**Explicación:** solo permanecen las filas que cumplen *ambas* condiciones: pedidos de las 10 h hechos por clientes con más de 10 pedidos.

### Ejercicio 10 · `products`: departamento 1 **y** `aisle_id` superior a 50

In [22]:
ej10 = products.loc[(products["department_id"] == 1) & (products["aisle_id"] > 50)]
mostrar(ej10, "Ejercicio 10")

**Ejercicio 10** → **1.618** registros · 4 columnas

,product_id,product_name,aisle_id,department_id
7,8,Cut Russet Potatoes Steam N' Mash,116,1
11,12,Chocolate Fudge Layer Cake,119,1
17,18,Pizza for One Suprema Frozen Pizza,79,1
45,46,Raisin Cinnamon Bagels 5 count,58,1
127,128,Organic Wild Blueberries,116,1


**Explicación:** productos del departamento 1 que además están en pasillos con identificador mayor a 50.

### Preguntas de análisis del Momento 4

In [23]:
md(f'''
**1. ¿Qué diferencia existe entre seleccionar columnas y filtrar filas?**
Seleccionar columnas decide **qué variables** se ven (Ejercicio 1: {ej1.shape[1]} columnas de {products.shape[1]}, pero las mismas {fmt(len(ej1))} filas).
Filtrar filas decide **qué registros** se ven según una condición (Ejercicio 2: {fmt(len(ej2))} de {fmt(len(products))} filas, con todas las columnas).

**2. ¿Qué ventajas ofrece `loc[]`?**
Permite elegir filas y columnas en una sola instrucción (`df.loc[filas, columnas]`), deja claro que se trabaja por **etiquetas y condiciones**,
combina bien con filtros lógicos y evita ambigüedades al asignar valores.

**3. ¿Qué ocurre con la cantidad de registros cuando se aplican varias condiciones?**
Con `&` el resultado solo puede **reducirse o mantenerse**, nunca crecer. Ejemplo: {fmt(len(ej3))} pedidos a las 10 h (Ejercicio 3) y {fmt(len(ej5))} con `order_number` > 10 (Ejercicio 5);
al exigir ambas condiciones quedan **{fmt(len(ej9))}** (Ejercicio 9). Con `|` ocurriría lo contrario: el resultado sería igual o mayor que cualquiera de las partes.

**4. ¿Por qué es importante utilizar correctamente los operadores lógicos?**
Porque cambian el resultado. Usar `|` en lugar de `&` devuelve filas que cumplen *alguna* condición y no *todas*. Además, cada condición debe ir entre paréntesis
(`&` tiene mayor precedencia que `>` o `==`) y se deben usar `&`, `|`, `~` en lugar de `and`, `or`, `not`; de lo contrario Pandas lanza un error.
''')


**1. ¿Qué diferencia existe entre seleccionar columnas y filtrar filas?**
Seleccionar columnas decide **qué variables** se ven (Ejercicio 1: 2 columnas de 4, pero las mismas 49.694 filas).
Filtrar filas decide **qué registros** se ven según una condición (Ejercicio 2: 4.008 de 49.694 filas, con todas las columnas).

**2. ¿Qué ventajas ofrece `loc[]`?**
Permite elegir filas y columnas en una sola instrucción (`df.loc[filas, columnas]`), deja claro que se trabaja por **etiquetas y condiciones**,
combina bien con filtros lógicos y evita ambigüedades al asignar valores.

**3. ¿Qué ocurre con la cantidad de registros cuando se aplican varias condiciones?**
Con `&` el resultado solo puede **reducirse o mantenerse**, nunca crecer. Ejemplo: 40.578 pedidos a las 10 h (Ejercicio 3) y 244.072 con `order_number` > 10 (Ejercicio 5);
al exigir ambas condiciones quedan **21.537** (Ejercicio 9). Con `|` ocurriría lo contrario: el resultado sería igual o mayor que cualquiera de las partes.

**4. ¿Por qué es importante utilizar correctamente los operadores lógicos?**
Porque cambian el resultado. Usar `|` en lugar de `&` devuelve filas que cumplen *alguna* condición y no *todas*. Además, cada condición debe ir entre paréntesis
(`&` tiene mayor precedencia que `>` o `==`) y se deben usar `&`, `|`, `~` en lugar de `and`, `or`, `not`; de lo contrario Pandas lanza un error.


---
## Momento 5. Obtención e interpretación de indicadores básicos

### 5.1 Diferenciar registros, pedidos y usuarios
Que una tabla tenga N filas **no** significa que haya N clientes. Se comprueba qué representa cada conteo:

In [24]:
registros_orders = orders.shape[0]
pedidos_unicos = orders["order_id"].nunique()
usuarios_unicos = orders["user_id"].nunique()

registros_op = order_products.shape[0]
pedidos_en_op = order_products["order_id"].nunique()

md(f'''
| Concepto | Valor | Cómo se obtiene |
|---|---|---|
| Registros (filas) en `orders` | **{fmt(registros_orders)}** | `orders.shape[0]` |
| Pedidos distintos (`order_id` únicos) | **{fmt(pedidos_unicos)}** | `orders["order_id"].nunique()` |
| Usuarios distintos (`user_id` únicos) | **{fmt(usuarios_unicos)}** | `orders["user_id"].nunique()` |
| Registros (filas) en `order_products` | **{fmt(registros_op)}** | `order_products.shape[0]` |
| Pedidos distintos en `order_products` | **{fmt(pedidos_en_op)}** | `order_products["order_id"].nunique()` |

Cada fila de `orders` representa un pedido, **no un cliente**: en promedio hay {fmt2(registros_orders / usuarios_unicos)} pedidos por usuario.
Cada fila de `order_products` representa **un artículo dentro de un pedido**, por eso esa tabla tiene muchas más filas que pedidos.
''')

if registros_orders != pedidos_unicos:
    md(f"⚠️ **Atención:** `orders` tiene {fmt(registros_orders)} filas pero solo {fmt(pedidos_unicos)} `order_id` distintos; "
       f"hay {fmt(registros_orders - pedidos_unicos)} filas con un `order_id` repetido.")


| Concepto | Valor | Cómo se obtiene |
|---|---|---|
| Registros (filas) en `orders` | **478.967** | `orders.shape[0]` |
| Pedidos distintos (`order_id` únicos) | **478.952** | `orders["order_id"].nunique()` |
| Usuarios distintos (`user_id` únicos) | **157.437** | `orders["user_id"].nunique()` |
| Registros (filas) en `order_products` | **4.545.007** | `order_products.shape[0]` |
| Pedidos distintos en `order_products` | **450.046** | `order_products["order_id"].nunique()` |

Cada fila de `orders` representa un pedido, **no un cliente**: en promedio hay 3,04 pedidos por usuario.
Cada fila de `order_products` representa **un artículo dentro de un pedido**, por eso esa tabla tiene muchas más filas que pedidos.


⚠️ **Atención:** `orders` tiene 478.967 filas pero solo 478.952 `order_id` distintos; hay 15 filas con un `order_id` repetido.

### 5.2 Cálculo de los indicadores

In [25]:
n_pedidos = orders.shape[0]
n_productos = products.shape[0]
prom_order_number = orders["order_number"].mean()
prom_dias = orders["days_since_prior_order"].mean()
n_dias_validos = orders["days_since_prior_order"].count()
n_reordered_1 = len(order_products[order_products["reordered"] == 1])
suma_reordered = order_products["reordered"].sum()

print("Registros en orders:               ", n_pedidos)
print("Registros en products:             ", n_productos)
print("Promedio de order_number:          ", prom_order_number)
print("Promedio de days_since_prior_order:", prom_dias, f"(sobre {n_dias_validos} valores disponibles)")
print("Registros con reordered == 1:      ", n_reordered_1)
print("Suma de reordered:                 ", suma_reordered)

Registros en orders:                478967
Registros en products:              49694
Promedio de order_number:           17.157808366755955
Promedio de days_since_prior_order: 11.101750979677794 (sobre 450148 valores disponibles)
Registros con reordered == 1:       2683838
Suma de reordered:                  2683838


**`describe()` de `order_hour_of_day`**

In [26]:
desc_hora = orders["order_hour_of_day"].describe()
desc_hora

count    478967.000000
mean         13.446676
std           4.224986
min           0.000000
25%          10.000000
50%          13.000000
75%          16.000000
max          23.000000
Name: order_hour_of_day, dtype: float64

**`describe()` de `add_to_cart_order`**

In [27]:
desc_carrito = order_products["add_to_cart_order"].describe()
desc_carrito

count    4.544171e+06
mean     8.351669e+00
std      7.080368e+00
min      1.000000e+00
25%      3.000000e+00
50%      6.000000e+00
75%      1.100000e+01
max      6.400000e+01
Name: add_to_cart_order, dtype: float64

### 5.3 Tabla de resultados: pregunta, valor e interpretación

In [28]:
n_op = len(order_products)
prop_reorden = n_reordered_1 / n_op
iguales = n_reordered_1 == suma_reordered

filas_ind = [
    ("¿Cuántos registros de pedidos contiene el archivo?",
     fmt(n_pedidos),
     f"Hay {fmt(n_pedidos)} filas en `orders`; cada una es un pedido. Corresponden a {fmt(usuarios_unicos)} usuarios distintos, no a {fmt(n_pedidos)} clientes."),
    ("¿Cuántos registros de productos contiene el catálogo?",
     fmt(n_productos),
     f"El catálogo tiene {fmt(n_productos)} productos, cada uno clasificado en un pasillo y un departamento."),
    ("¿Cuál es el promedio de `order_number`?",
     fmt2(prom_order_number),
     f"En promedio, un pedido es el n.º {fmt2(prom_order_number)} de su usuario: los pedidos provienen en su mayoría de clientes con historial de compra."),
    ("¿Cuál es el promedio de `days_since_prior_order` (valores disponibles)?",
     f"{fmt2(prom_dias)} días",
     f"Entre un pedido y el siguiente transcurren en promedio {fmt2(prom_dias)} días. Se calculó con {fmt(n_dias_validos)} valores; los {fmt(n_pedidos - n_dias_validos)} ausentes no entran en el promedio."),
    ("¿Cuántos registros de artículos tienen `reordered` igual a 1?",
     fmt(n_reordered_1),
     f"{fmt(n_reordered_1)} de {fmt(n_op)} artículos ({fmt2(100 * prop_reorden)} %) son recompras."),
    ("¿Cuál es la suma de la columna `reordered`?",
     fmt(suma_reordered),
     ("Coincide con el conteo anterior: al ser una columna de 0 y 1, sumar equivale a contar los 1."
      if iguales else "No coincide con el conteo anterior, lo que indica que la columna tiene valores distintos de 0 y 1.")),
    ("¿Qué características estadísticas presenta `order_hour_of_day`?",
     f"media {fmt2(desc_hora['mean'])}; mín {desc_hora['min']:.0f}; máx {desc_hora['max']:.0f}",
     f"Los pedidos se hacen entre las {desc_hora['min']:.0f} h y las {desc_hora['max']:.0f} h. La mitad ocurre antes de las {desc_hora['50%']:.0f} h (mediana) y el 25 %–75 % central entre las {desc_hora['25%']:.0f} h y las {desc_hora['75%']:.0f} h."),
    ("¿Qué características estadísticas presenta `add_to_cart_order`?",
     f"media {fmt2(desc_carrito['mean'])}; mediana {desc_carrito['50%']:.0f}; máx {desc_carrito['max']:.0f}",
     f"La posición media de un artículo en el carrito es {fmt2(desc_carrito['mean'])} y el carrito más largo registrado llega a la posición {desc_carrito['max']:.0f}. "
     f"`count` = {fmt(desc_carrito['count'])} frente a {fmt(n_op)} registros: la diferencia son valores ausentes."),
]

tabla_md = "| Pregunta | Valor obtenido | Interpretación |\n|---|---|---|\n"
tabla_md += "\n".join(f"| {p} | **{v}** | {i} |" for p, v, i in filas_ind)
md(tabla_md)

| Pregunta | Valor obtenido | Interpretación |
|---|---|---|
| ¿Cuántos registros de pedidos contiene el archivo? | **478.967** | Hay 478.967 filas en `orders`; cada una es un pedido. Corresponden a 157.437 usuarios distintos, no a 478.967 clientes. |
| ¿Cuántos registros de productos contiene el catálogo? | **49.694** | El catálogo tiene 49.694 productos, cada uno clasificado en un pasillo y un departamento. |
| ¿Cuál es el promedio de `order_number`? | **17,16** | En promedio, un pedido es el n.º 17,16 de su usuario: los pedidos provienen en su mayoría de clientes con historial de compra. |
| ¿Cuál es el promedio de `days_since_prior_order` (valores disponibles)? | **11,10 días** | Entre un pedido y el siguiente transcurren en promedio 11,10 días. Se calculó con 450.148 valores; los 28.819 ausentes no entran en el promedio. |
| ¿Cuántos registros de artículos tienen `reordered` igual a 1? | **2.683.838** | 2.683.838 de 4.545.007 artículos (59,05 %) son recompras. |
| ¿Cuál es la suma de la columna `reordered`? | **2.683.838** | Coincide con el conteo anterior: al ser una columna de 0 y 1, sumar equivale a contar los 1. |
| ¿Qué características estadísticas presenta `order_hour_of_day`? | **media 13,45; mín 0; máx 23** | Los pedidos se hacen entre las 0 h y las 23 h. La mitad ocurre antes de las 13 h (mediana) y el 25 %–75 % central entre las 10 h y las 16 h. |
| ¿Qué características estadísticas presenta `add_to_cart_order`? | **media 8,35; mediana 6; máx 64** | La posición media de un artículo en el carrito es 8,35 y el carrito más largo registrado llega a la posición 64. `count` = 4.544.171 frente a 4.545.007 registros: la diferencia son valores ausentes. |

---
## Momento 6. Resolución del problema y presentación del producto
Síntesis individual que responde la pregunta orientadora. Todas las cifras se calculan a partir de los DataFrames cargados.

In [29]:
# --- Respuestas a las tres preguntas de análisis formuladas en el Momento 1
r1 = orders["days_since_prior_order"].mean()
r1_med = orders["days_since_prior_order"].median()
r2_n = len(order_products[order_products["reordered"] == 1])
r2_prop = r2_n / len(order_products)
r3 = len(orders.loc[(orders["order_hour_of_day"] == 10) & (orders["order_number"] > 10)])
r3_pct = 100 * r3 / len(orders)

# --- Datos para hallazgos y limitaciones
items_por_pedido = len(order_products) / order_products["order_id"].nunique()
hora_media = orders["order_hour_of_day"].mean()
nulos_txt = "; ".join(
    f"`{r.DataFrame}.{r.columna}` ({fmt(r.ausentes)})" for r in calidad[calidad["ausentes"] > 0].itertuples(index=False)
) or "ninguna columna"
dup = len(orders) - orders["order_id"].nunique()
tiene_missing = bool(aisles["aisle"].eq("missing").any() or departments["department"].eq("missing").any())

mitad = "más de la mitad" if r2_prop > 0.5 else "menos de la mitad"

md(f'''
### 6.1 Las tres preguntas de análisis y sus respuestas
1. **¿Cada cuántos días, en promedio, vuelven a comprar los clientes?**
   En promedio **{fmt2(r1)} días** entre pedidos (mediana: {fmt2(r1_med)} días), calculado solo con los pedidos que tienen un pedido anterior.
2. **¿Qué proporción de los artículos son recompras?**
   **{fmt2(100 * r2_prop)} %** ({fmt(r2_n)} de {fmt(len(order_products))} registros de `order_products`), es decir, {mitad} de los artículos ya se habían comprado antes.
3. **¿Cuántos pedidos se realizan a las 10 horas y son de clientes con más de 10 pedidos?**
   **{fmt(r3)}** pedidos, equivalentes al {fmt2(r3_pct)} % de los registros de `orders`.

### 6.2 Principales hallazgos sobre pedidos y productos
- **Pedidos:** `orders` tiene {fmt(orders.shape[0])} registros de {fmt(orders["user_id"].nunique())} usuarios distintos. El promedio de `order_number` es {fmt2(orders["order_number"].mean())}
  y la hora media de compra es las {fmt2(hora_media)} h.
- **Productos:** el catálogo tiene {fmt(products.shape[0])} productos repartidos en {fmt(aisles.shape[0])} pasillos y {fmt(departments.shape[0])} departamentos.
- **Detalle de compras:** `order_products` tiene {fmt(order_products.shape[0])} registros; en promedio cada pedido incluye {fmt2(items_por_pedido)} artículos.

### 6.3 Cómo la indexación y los filtros permitieron consultar información relevante
Seleccionar columnas con `loc[:, [...]]` permitió ver solo los identificadores y nombres de productos. Los filtros lógicos aislaron subconjuntos concretos:
pedidos a una hora determinada, de clientes recurrentes (`order_number` > 10), con más de 15 días entre compras, recompras y primeros productos del carrito.
Combinar condiciones con `&` redujo los {fmt(len(ej3))} pedidos de las 10 h a **{fmt(len(ej9))}**, lo que permite responder preguntas más específicas sin recorrer fila por fila.

### 6.4 Limitaciones identificadas en los conjuntos de datos
- **Valores ausentes:** {nulos_txt}. Los promedios y descripciones estadísticas usan menos registros que el total de filas.
- **Filas no son clientes:** `orders` tiene {fmt(orders.shape[0])} filas pero {fmt(orders["user_id"].nunique())} usuarios; sin distinguir esto se sobrestimaría el número de clientes.
- **Pedidos duplicados:** {"hay " + fmt(dup) + " filas de `orders` con un `order_id` repetido, que convendría revisar en una fase de limpieza." if dup > 0 else "no se detectaron `order_id` repetidos en `orders`."}
- **Sin diccionario de códigos:** el archivo no indica a qué día corresponde cada valor de `order_dow`.
- **Categorías «missing»:** {"los catálogos de pasillos y departamentos incluyen una categoría llamada `missing`, es decir, productos sin clasificación real." if tiene_missing else "no se encontraron categorías `missing` en los catálogos."}
- **Sin fechas ni cantidades:** los datos no incluyen la fecha de compra, el precio ni la cantidad de cada artículo, lo que limita el tipo de análisis posible.

### 6.5 Tres conclusiones sustentadas en los resultados
1. **Los clientes compran con regularidad:** el promedio de {fmt2(r1)} días entre pedidos y un `order_number` medio de {fmt2(orders["order_number"].mean())} muestran una base de clientes que vuelve a comprar.
2. **La recompra es {"un componente central" if r2_prop > 0.5 else "una parte importante"} del consumo:** {fmt2(100 * r2_prop)} % de los artículos incluidos en pedidos son productos ya comprados antes.
3. **Los filtros lógicos permiten segmentar a los clientes:** con dos condiciones simultáneas se identificaron {fmt(r3)} pedidos de clientes recurrentes a las 10 h, un grupo de interés para campañas o análisis posteriores.

### 6.6 Respuesta a la pregunta orientadora
Con `pd.read_csv(sep=";")` se cargaron los cinco archivos; `shape`, `dtypes` e `info()` revelaron su estructura y sus valores ausentes; `loc[]` con filtros lógicos extrajo
subconjuntos específicos; y `mean()`, `sum()` y `describe()` entregaron indicadores sobre pedidos, recompras y horarios. Así, Python y Pandas permiten pasar de archivos CSV sin explorar
a un conocimiento verificable de cómo compran los clientes de Instacart.
''')


### 6.1 Las tres preguntas de análisis y sus respuestas
1. **¿Cada cuántos días, en promedio, vuelven a comprar los clientes?**
   En promedio **11,10 días** entre pedidos (mediana: 7,00 días), calculado solo con los pedidos que tienen un pedido anterior.
2. **¿Qué proporción de los artículos son recompras?**
   **59,05 %** (2.683.838 de 4.545.007 registros de `order_products`), es decir, más de la mitad de los artículos ya se habían comprado antes.
3. **¿Cuántos pedidos se realizan a las 10 horas y son de clientes con más de 10 pedidos?**
   **21.537** pedidos, equivalentes al 4,50 % de los registros de `orders`.

### 6.2 Principales hallazgos sobre pedidos y productos
- **Pedidos:** `orders` tiene 478.967 registros de 157.437 usuarios distintos. El promedio de `order_number` es 17,16
  y la hora media de compra es las 13,45 h.
- **Productos:** el catálogo tiene 49.694 productos repartidos en 134 pasillos y 21 departamentos.
- **Detalle de compras:** `order_products` tiene 4.545.007 registros; en promedio cada pedido incluye 10,10 artículos.

### 6.3 Cómo la indexación y los filtros permitieron consultar información relevante
Seleccionar columnas con `loc[:, [...]]` permitió ver solo los identificadores y nombres de productos. Los filtros lógicos aislaron subconjuntos concretos:
pedidos a una hora determinada, de clientes recurrentes (`order_number` > 10), con más de 15 días entre compras, recompras y primeros productos del carrito.
Combinar condiciones con `&` redujo los 40.578 pedidos de las 10 h a **21.537**, lo que permite responder preguntas más específicas sin recorrer fila por fila.

### 6.4 Limitaciones identificadas en los conjuntos de datos
- **Valores ausentes:** `orders.days_since_prior_order` (28.819); `products.product_name` (1.258); `order_products.add_to_cart_order` (836). Los promedios y descripciones estadísticas usan menos registros que el total de filas.
- **Filas no son clientes:** `orders` tiene 478.967 filas pero 157.437 usuarios; sin distinguir esto se sobrestimaría el número de clientes.
- **Pedidos duplicados:** hay 15 filas de `orders` con un `order_id` repetido, que convendría revisar en una fase de limpieza.
- **Sin diccionario de códigos:** el archivo no indica a qué día corresponde cada valor de `order_dow`.
- **Categorías «missing»:** los catálogos de pasillos y departamentos incluyen una categoría llamada `missing`, es decir, productos sin clasificación real.
- **Sin fechas ni cantidades:** los datos no incluyen la fecha de compra, el precio ni la cantidad de cada artículo, lo que limita el tipo de análisis posible.

### 6.5 Tres conclusiones sustentadas en los resultados
1. **Los clientes compran con regularidad:** el promedio de 11,10 días entre pedidos y un `order_number` medio de 17,16 muestran una base de clientes que vuelve a comprar.
2. **La recompra es un componente central del consumo:** 59,05 % de los artículos incluidos en pedidos son productos ya comprados antes.
3. **Los filtros lógicos permiten segmentar a los clientes:** con dos condiciones simultáneas se identificaron 21.537 pedidos de clientes recurrentes a las 10 h, un grupo de interés para campañas o análisis posteriores.

### 6.6 Respuesta a la pregunta orientadora
Con `pd.read_csv(sep=";")` se cargaron los cinco archivos; `shape`, `dtypes` e `info()` revelaron su estructura y sus valores ausentes; `loc[]` con filtros lógicos extrajo
subconjuntos específicos; y `mean()`, `sum()` y `describe()` entregaron indicadores sobre pedidos, recompras y horarios. Así, Python y Pandas permiten pasar de archivos CSV sin explorar
a un conocimiento verificable de cómo compran los clientes de Instacart.
